# When "Probably" Isn't the Same as "Definitely"

**Scenario.** A new analyst joining Pitch & Slate Music's Growth team has a question: why
don't we just run promotion campaigns for every genre with a hit rate above 50%?
Those are the likely winners. You've been asked to show her why the 50% threshold
is the wrong one — using track data and a bit of arithmetic.

Each genre targets a distinct listener audience, so promotional slots aren't
interchangeable: the decision per genre is simply take it or leave it.



## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH        = "data/spotify_tracks_sample.csv"

HIT_THRESHOLD    = 50    # Spotify popularity ≥ this = "hit"
NET_GAIN_K       = 15    # net gain per campaign if track hits ($K)
COST_K           = 8     # cost per campaign if track misses ($K)
BINARY_THRESHOLD = 0.50  # Pitch & Slate Music's current rule: hit rate > 50% → run campaign
N_PER_GENRE      = 100   # promotion slots per genre per season

RNG = np.random.default_rng(42)

## 1. Load track data and compute genre hit rates

Load the track sample, classify each track as a hit or miss based on
`track_popularity`, and compute each genre's hit rate from the data.

In [ ]:
# TODO: Read the CSV into a DataFrame called `tracks`.
tracks = ...

# TODO: Add a column `is_hit`: 1 if track_popularity >= HIT_THRESHOLD, else 0.

# TODO: Compute `genre_stats` — a DataFrame with one row per genre containing
#       at least `hit_rate` (the proportion of tracks that are hits) and `n` (count).
#       Sort by hit_rate ascending.
genre_stats = ...

genre_stats

## 2. Compute the break-even hit rate

Find the minimum hit rate at which a campaign has non-negative expected value,
given the payoffs defined in Setup.

In [ ]:
# TODO: Compute BREAK_EVEN from NET_GAIN_K and COST_K. Do not hardcode it.
BREAK_EVEN = ...

print(f"Break-even hit rate:         {BREAK_EVEN:.1%}")
print(f"Pitch & Slate Music's binary threshold: {BINARY_THRESHOLD:.0%}")
print()

# TODO: Add two boolean columns to genre_stats:
#   `binary_runs` — True if hit_rate > BINARY_THRESHOLD
#   `profitable_runs`     — True if hit_rate > BREAK_EVEN
genre_stats["binary_runs"] = ...
genre_stats["profitable_runs"]     = ...

# TODO: Print the disagreement zone — genres EV says run but binary says skip.
disagreement = ...
print("Disagreement zone (EV says run, binary says skip):")
disagreement[["playlist_genre", "hit_rate"]]

## 3. The certainty illusion — simulate 100 Pop campaigns

Pop has a 54% hit rate — the only genre above the 50% binary threshold.
Pitch & Slate Music's rule treats this as "it'll work." Simulate 100 Pop promotion campaigns
using `RNG.binomial(1, rate, N_PER_GENRE)` and count how many miss.

In [ ]:
# TODO: Extract Pop's hit rate from genre_stats.
pop_rate = ...

# TODO: Simulate N_PER_GENRE Pop campaigns.
pop_outcomes = ...
pop_hits   = ...
pop_misses = ...

print(f"Pop hit rate: {pop_rate:.1%}  (binary thinking: 'above 50% — it\'ll work')")
print(f"\nSimulated result across {N_PER_GENRE} campaigns:")
print(f"  Hits:   {pop_hits}")
print(f"  Misses: {pop_misses}")

# TODO: In 1 sentence, explain what the miss count reveals about treating 54% as certainty.

## 4. The missed opportunity — simulate 100 R&B campaigns

R&B has a 37% hit rate — below 50%, so Pitch & Slate Music skips it. But 37% is not zero.
Simulate 100 R&B campaigns and count how many hit. Then compute the expected
value of one R&B campaign to show whether skipping it was the right call.

In [ ]:
# TODO: Extract R&B's hit rate from genre_stats.
rb_rate = ...

# TODO: Simulate N_PER_GENRE R&B campaigns. Same pattern as step 3.
rb_outcomes = ...
rb_hits   = ...
rb_misses = ...

# TODO: Compute the expected value per R&B campaign and the total expected profit
#       Pitch & Slate Music left on the table by skipping 100 R&B campaigns.
rb_avg_per_campaign = ...

print(f"R&B hit rate: {rb_rate:.1%}  (binary thinking: 'below 50% — won't happen')")
print(f"\nSimulated result across {N_PER_GENRE} campaigns:")
print(f"  Hits:   {rb_hits}")
print(f"  Misses: {rb_misses}")
print(f"\nExpected value per R&B campaign: ${rb_avg_per_campaign:.1f}K")

# TODO: In 1 sentence, explain what the result reveals about treating 37% as impossible.

## 5. One simulated season per strategy

A "season" is N_PER_GENRE campaigns per genre that the strategy runs.
Implement `simulate_season`, then compare one simulated season for each strategy.

In [ ]:
# TODO: Implement simulate_season(hit_rates, n_per_genre).
#       Use RNG.binomial to draw campaign outcomes, then compute total profit.
def simulate_season(hit_rates: np.ndarray, n_per_genre: int = N_PER_GENRE) -> float:
    """Return total season profit ($K) for a set of genres with given hit rates."""
    ...


# TODO: Run simulate_season for each strategy and print the dollar difference.
binary_hit_rates = genre_stats.loc[genre_stats["binary_runs"], "hit_rate"].values
payoff_hit_rates     = genre_stats.loc[genre_stats["profitable_runs"],     "hit_rate"].values

binary_season = simulate_season(binary_hit_rates)
payoff_season     = simulate_season(payoff_hit_rates)

print(f"One simulated season:")
print(f"  Binary strategy:   ${binary_season:,.0f}K")
print(f"  Payoff-aware strategy: ${payoff_season:,.0f}K")
print(f"  Difference:        ${payoff_season - binary_season:+,.0f}K")

## 6. Analytical expected profit — why the gap is systematic

One season is one random draw. Compute the expected profit analytically —
no simulation needed. For each genre, combine its hit rate with the payoff
constants to get an expected profit per campaign, then scale up to a full season.

Implement `average_season_profit` and compute it for each strategy.

In [ ]:
# TODO: Implement average_season_profit(hit_rates, n_per_genre).
#       No randomness — just multiply hit rates through the payoff formula.
def average_season_profit(hit_rates: np.ndarray, n_per_genre: int = N_PER_GENRE) -> float:
    """Return expected season profit ($K) — analytical, no randomness."""
    ...


binary_ev = average_season_profit(binary_hit_rates)
ev_ev     = average_season_profit(payoff_hit_rates)

print(f"Binary strategy expected season profit:    ${binary_ev:,.0f}K")
print(f"Payoff-aware strategy expected season profit:  ${ev_ev:,.0f}K")
print(f"Systematic gap:                            ${ev_ev - binary_ev:,.0f}K per season")

## 7. Takeaway

*TODO: Write 2–3 sentences. Use specific numbers from steps 3–6. Explain:*
- *What the coin-flip simulations (steps 3–4) revealed about binary thinking.*
- *Why the analytical gap (step 6) matters more than the single-season result (step 5).*